In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('../../data/merged/merged_final.csv', low_memory=False)

## Price Features

In [3]:
df["price_change_pct"] = df["price_change"] / df["last_years_price"]

# Replace inf with NaN
df["price_change_pct"] = df["price_change_pct"].replace([np.inf, -np.inf], np.nan)

# Add flag
df["no_last_price_flag"] = df["last_years_price"].isna() | (df["last_years_price"] == 0)
df["no_last_price_flag"] = df["no_last_price_flag"].astype(int)

In [4]:
df = df.drop("last_years_price",axis=1)

## Call Features

In [5]:
df["engagement_ratio"] = df["inbound_calls"] / (df["total_calls"] + 1)
df["is_no_call_customer"] = (df["total_calls"] == 0).astype(int)
df["is_high_call_customer"] = (df["total_calls"] > 3).astype(int)
df["outbound_ratio"] = df["outbound_calls"] / (df["total_calls"] + 1)

## Flag Features

In [6]:
leakage_cols = [
    "renewal_confirmed_flag",
    "prospect_outcome",
    "closed_date",
    "customer_renewal_response_category",
    "agent_renewal_pitch_category",
    "price_impact_renewal_flag",
    "discount_offered_flag"
]

safe_flags = [
    "serious_complaint_flag",
    "other_complaint_flag",
    "competitor_mentioned_flag",
    "switching_intent_flag",
    "price_discussed_flag",
    "discount_requested_flag"
]

drop_cols = [
    # raw flags (optional, after aggregation)
    "serious_complaint_flag",
    "other_complaint_flag",
    "competitor_mentioned_flag",
    "switching_intent_flag",
    "price_discussed_flag",
    "discount_requested_flag"
]

In [7]:
df = df.drop(columns=leakage_cols)

In [8]:
#type printer
for i in safe_flags:
    print(f'{i}: {df[i].dtype}')

serious_complaint_flag: int64
other_complaint_flag: int64
competitor_mentioned_flag: int64
switching_intent_flag: int64
price_discussed_flag: int64
discount_requested_flag: int64


In [9]:
new_features = {}

# FLAGS
new_features["total_flags"] = df[safe_flags].sum(axis=1)
new_features["has_complaint"] = (
    df["serious_complaint_flag"] |
    df["other_complaint_flag"]
).astype(int)

new_features["switching_risk"] = (
    df["competitor_mentioned_flag"] |
    df["switching_intent_flag"]
).astype(int)

new_features["price_issue"] = (
    df["price_discussed_flag"] |
    df["discount_requested_flag"]
).astype(int)

# Add all at once
df = pd.concat([df, pd.DataFrame(new_features)], axis=1)

In [10]:
df["total_flags"].value_counts()

total_flags
0    43825
1     1667
2      598
3      218
4       17
5        1
Name: count, dtype: int64

In [11]:
df["flag_risk_level"] = pd.cut(
    df["total_flags"],
    bins=[-1, 0, 2, 10],
    labels=["low", "medium", "high"]
)

In [12]:
df["overall_score"] = (
    df["status_scores"] +
    df["total_renewal_score_new"]
) / 2

In [13]:
crm_cols = [
    c for c in df.columns
    if c.startswith("crm_")
    and "score" not in c
    and "sentiment" not in c
]

df[crm_cols] = (
    df[crm_cols]
    .replace({"Unknown": np.nan, "No_Call": np.nan})
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .clip(0, 1)
)

In [14]:
crm_progress_cols = [
    "crm_accreditation_completed",
    "crm_timely_completion",
    "crm_progress_towards_accreditation"
]

df["crm_progress_score"] = df[crm_progress_cols].sum(axis=1)

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\2165956057.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_progress_score"] = df[crm_progress_cols].sum(axis=1)


In [15]:
df["crm_complaint_behavior"] = (
    df["crm_customer_complained"] +
    df["crm_negative_customer_experience"] +
    df["crm_refund_mentioned"]
)

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\1926831446.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_complaint_behavior"] = (


In [16]:
crm_delay_cols = [
    "crm_delays_in_accreditation",
    "crm_platform_issues_raised",
    "crm_accreditation_issues"
]

df["crm_friction_score"] = df[crm_delay_cols].sum(axis=1)

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\2054703774.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_friction_score"] = df[crm_delay_cols].sum(axis=1)


In [17]:
df["crm_financial_risk"] = (
    df["crm_financial_hardship_mentioned"] +
    df["crm_customer_payment_intention"]
)

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\2798767993.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_financial_risk"] = (


In [18]:
crm_negative_cols = [
    "crm_customer_complained",
    "crm_negative_customer_experience",
    "crm_dissatisfaction_with_support",
    "crm_dissatisified_with_renewal_price"
]

df["crm_negative_score"] = df[crm_negative_cols].sum(axis=1)

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\3558067684.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_negative_score"] = df[crm_negative_cols].sum(axis=1)


In [19]:
df["crm_exit_signal"] = (
    df["crm_refund_mentioned"] +
    df["crm_membership_overdue"]
)

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\2385528377.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_exit_signal"] = (


In [20]:
df["crm_sentiment"] = df["crm_contractor_sentiment_score_num"]

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\286465759.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_sentiment"] = df["crm_contractor_sentiment_score_num"]


In [21]:
df["crm_engagement"] = (
    df["crm_contractor_engagement"] +
    df["crm_agent_chased_contractor"]
)

C:\Users\AtharvaRana\AppData\Local\Temp\ipykernel_19236\1268523028.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crm_engagement"] = (


In [22]:
raw_crm_cols = [
    'crm_accreditation_completed',
    'crm_timely_completion',
    'crm_progress_towards_accreditation',
    'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave',
    'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned',
    'crm_customer_payment_intention',
    'crm_competitors_mentioned',
    'crm_platform_issues_raised',
    'crm_agent_chased_contractor',
    'crm_accreditation_issues',
    'crm_membership_overdue',
    'crm_dissatisified_with_renewal_price',
    'crm_customer_complained',
    'crm_refund_mentioned',
    'crm_negative_customer_experience',
    'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
    'crm_membership_level',
    'crm_agent_chase_count_agg',
    'crm_contractor_sentiment',
]
keep_crm = [
    "crm_progress_score",
    "crm_friction_score",
    "crm_financial_risk",
    "crm_negative_score",
    "crm_exit_signal",
    "crm_engagement",
    "crm_sentiment"
]

In [23]:
df.drop(columns=raw_crm_cols, inplace=True)

In [24]:
df["call_complaint_intensity"] = df["total_calls"] * df["has_complaint"]
df["risk_engagement"] = df["total_flags"] * df["engagement_ratio"]
df["price_shock"] = df["price_change_pct"] * df["amount"]
df["price_per_connection"] = df["amount"] / (df["#_of_connection"] + 1)
df["call_intensity"] = df["total_calls"] / (df["tenure_years"] + 1)
df["inbound_ratio"] = df["inbound_calls"] / (df["total_calls"] + 1)


df["silent_customer_risk"] = (
    (df["total_calls"] == 0) & (df["crm_engagement"] == 0)
).astype(int)

df["complaint_price_risk"] = df["total_flags"] * df["price_change_pct"]
df["crm_call_risk"] = df["crm_negative_score"] * df["total_calls"]
df["switching_engagement_risk"] = df["switching_risk"] * df["engagement_ratio"]

df["customer_age_days"] = (
    pd.to_datetime(df["prospect_renewal_date"]) -
    pd.to_datetime(df["registration_date"])
).dt.days

df["tenure_bucket"] = pd.cut(
    df["tenure_years"],
    bins=[0, 2, 5, 10, 50],
    labels=["new", "mid", "loyal", "very_loyal"]
)

df["renewal_month_num"] = pd.to_datetime(df["prospect_renewal_date"]).dt.month

df["customer_age_days"] = (
    pd.to_datetime(df["prospect_renewal_date"]) -
    pd.to_datetime(df["registration_date"])
).dt.days

In [25]:
drop_cols = [
    "co_ref",                  # ID
    "registration_date",       # will extract features first (see below)
    "prospect_renewal_date",   # will extract features first
    "renewal_month"            # redundant (we'll recreate better)
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [26]:
leakage_cols = [
    # direct outcome
    "prospect_outcome",

    # decision-stage flags
    "renewal_confirmed_flag",
    "agent_initiated_renewal_flag",

    # pricing negotiation
    "discount_offered_flag",
    "price_impact_renewal_flag",

    # final interaction summaries
    "customer_renewal_response_category",
    "agent_renewal_pitch_category",
    "agent_response_category",

    # time leakage
    "closed_date",
    "days_to_renewal",

    # high-risk CC leakage
    "cc_refund_discussed",
    "cc_pricing_sentiment_impact",
    "cc_contractor_suggest_leave",

    #Leakage Score Cols
    "status_scores",
    "total_renewal_score_new",
    "overall_score",
    "auto_renewal_score",
    "renewal_score_at_release",

    "prospect_status",
    "complaint_category",
    "customer_reaction_category",
    "payment_method",

    "current_auto_renewal_flag",
    "payment_timeframe",
    "sustainability_score",


    "total_net_paid",
    "proforma_approved_lists",
    "current_anchorings",
    "anchoring_score",
    "tenure_scores"




]

df.drop(columns=[c for c in leakage_cols if c in df.columns], inplace=True)

In [27]:
df.to_csv("../../data/ml_data/Feature_Engineered.csv", index=False)
print("Saved dataset")

Saved dataset
